# Лабораторная работа №4. Вариант 17
## Построение ETL-конвейера средствами Dask

**Цель:** получить практические навыки работы с библиотекой Dask для построения базовых ETL-конвейеров при обработке больших массивов данных.

**Датасет:** UK Property Price official data 1995-202304 (~4.6 ГБ)

## Описание датасета

Датасет содержит данные о сделках с недвижимостью в Англии и Уэльсе, опубликованные HM Land Registry (Земельный кадастр Великобритании) за период с 1995 по апрель 2023 года.

| Столбец | Описание |
|---------|----------|
| **transaction_id** | Уникальный идентификатор сделки (GUID) |
| **price** | Цена продажи объекта недвижимости в фунтах стерлингов |
| **date_of_transfer** | Дата совершения сделки |
| **postcode** | Почтовый индекс объекта недвижимости |
| **property_type** | Тип недвижимости: **D** — отдельный дом (Detached), **S** — двухквартирный дом (Semi-Detached), **T** — дом рядовой застройки (Terraced), **F** — квартира (Flat/Maisonette), **O** — другое (Other) |
| **old_new** | Возраст постройки: **Y** — новостройка, **N** — вторичное жилье |
| **duration** | Тип владения: **F** — полное право собственности (Freehold), **L** — аренда (Leasehold) |
| **paon** | Основной адресный элемент (номер/название дома) |
| **saon** | Дополнительный адресный элемент (номер квартиры и т.п.) |
| **street** | Название улицы |
| **locality** | Район/местность |
| **town_city** | Город |
| **district** | Административный район |
| **county** | Графство |
| **ppd_category** | Категория данных: **A** — стандартная сделка, **B** — дополнительная (нерыночная) сделка |
| **record_status** | Статус записи: **A** — добавлена, **C** — изменена, **D** — удалена |

## Установка зависимостей

In [ ]:
!pip install "dask[complete]" graphviz pyarrow matplotlib

## Задание 4.1. Построение ETL-пайплайна
### Шаг 1. Extract (Извлечение данных)

Настраиваем локальный кластер Dask и лениво загружаем датасет. Данные на этом этапе не загружаются в оперативную память целиком.

In [ ]:
import dask.dataframe as dd
from dask.distributed import Client
from dask.diagnostics import ProgressBar
import dask
import matplotlib.pyplot as plt

# Инициализация клиента Dask (оптимизированные настройки)
client = Client(n_workers=2, threads_per_worker=2, processes=True)
display(client)

In [ ]:
# Определяем имена столбцов для UK Land Registry Price Paid Data
column_names = [
    'transaction_id',
    'price',
    'date_of_transfer',
    'postcode',
    'property_type',
    'old_new',
    'duration',
    'paon',
    'saon',
    'street',
    'locality',
    'town_city',
    'district',
    'county',
    'ppd_category',
    'record_status'
]

# Чтение CSV файла (Extract) - ленивая загрузка
df = dd.read_csv(
    '202304.csv',
    header=None,
    names=column_names,
    dtype='str',
    blocksize='64MB'
)

# Просмотр структуры (метаданных) датасета - фактического чтения еще не произошло
df

In [ ]:
# Количество разделов (partitions)
print(f"Количество разделов (partitions): {df.npartitions}")
print(f"Столбцы: {list(df.columns)}")
print(f"Типы данных:\n{df.dtypes}")

### Шаг 2. Transform (Трансформация и очистка данных)

Проводим профилирование качества данных: вычисляем процент пропущенных значений. Используем `.compute()` только для получения финального результата.

In [ ]:
# Подсчет пропущенных значений (построение графа вычислений)
missing_values = df.isnull().sum()

# Вычисление процента пропусков
mysize = df.index.size
missing_count = ((missing_values / mysize) * 100)

# Запуск реальных вычислений только для агрегированной статистики
with ProgressBar():
    missing_count_percent = missing_count.compute()

print("Процент пропущенных значений по столбцам:")
print(missing_count_percent)

In [ ]:
# Формирование списка столбцов, где пропусков > 60%
columns_to_drop = list(missing_count_percent[missing_count_percent > 60].index)
print("Удаляемые столбцы (пропусков > 60%):", columns_to_drop)

In [ ]:
# ОПТИМИЗАЦИЯ: Ленивое удаление столбцов без вызова .compute()!
# Метод .drop() лишь записывает правило "не читать эти столбцы в будущем".
df_dropped = df.drop(columns=columns_to_drop)

print(f"Столбцов до очистки: {len(df.columns)}")
print(f"Столбцов после очистки: {len(df_dropped.columns)}")
print(f"Оставшиеся столбцы: {list(df_dropped.columns)}")

# Проверка результата (вычислит только первые 5 строк первого блока)
df_dropped.head()

### Шаг 3. Load (Загрузка / Сохранение результатов)

Сохраняем очищенный Dask DataFrame на диск в формате Parquet. Это позволяет записать огромный массив данных по частям.

In [ ]:
# Сохранение результатов трансформации в формате Parquet (Load)
with ProgressBar():
    df_dropped.to_parquet('cleaned_dataset.parquet', engine='pyarrow')

print("Данные успешно сохранены в cleaned_dataset.parquet")

In [ ]:
# Проверка: читаем сохраненный parquet
df_check = dd.read_parquet('cleaned_dataset.parquet')
print(f"Разделов: {df_check.npartitions}")
print(f"Столбцов: {len(df_check.columns)}")
df_check.head()

---
## Анализ данных с визуализацией

Проведем анализ датасета и построим 4 графика для наглядного представления основных характеристик рынка недвижимости Великобритании.

In [ ]:
# Подготовка данных для визуализации
df_analysis = df_dropped.copy()
df_analysis['price'] = df_analysis['price'].astype(float)
df_analysis['year'] = df_analysis['date_of_transfer'].str[:4].astype(int)

In [ ]:
# График 1: Количество сделок по годам
with ProgressBar():
    transactions_by_year = df_analysis.groupby('year')['transaction_id'].count().compute()

fig, ax = plt.subplots(figsize=(14, 5))
transactions_by_year.sort_index().plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Quantity of real estate transactions by year', fontsize=14)
ax.set_xlabel('Year')
ax.set_ylabel('Number of transactions')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# График 2: Средняя цена недвижимости по годам
with ProgressBar():
    avg_price_by_year = df_analysis.groupby('year')['price'].mean().compute()

fig, ax = plt.subplots(figsize=(14, 5))
avg_price_by_year.sort_index().plot(kind='line', ax=ax, marker='o', color='darkgreen', linewidth=2)
ax.set_title('Average property price by year', fontsize=14)
ax.set_xlabel('Year')
ax.set_ylabel('Average price (GBP)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# График 3: Распределение сделок по типу недвижимости (круговая диаграмма)
type_labels = {
    'D': 'Detached',
    'S': 'Semi-Detached',
    'T': 'Terraced',
    'F': 'Flat/Maisonette',
    'O': 'Other'
}

with ProgressBar():
    by_type = df_analysis.groupby('property_type')['transaction_id'].count().compute()

fig, ax = plt.subplots(figsize=(8, 6))
labels = [type_labels.get(t, t) for t in by_type.index]
colors = ['#ff9999', '#66b3ff', '#99ff99', '#ffcc99', '#c2c2f0']
ax.pie(by_type.values, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
ax.set_title('Distribution of transactions by property type', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# График 4: Средняя цена по типу недвижимости (столбчатая диаграмма)
with ProgressBar():
    avg_price_by_type = df_analysis.groupby('property_type')['price'].mean().compute()

fig, ax = plt.subplots(figsize=(8, 5))
labels = [type_labels.get(t, t) for t in avg_price_by_type.index]
bars = ax.bar(labels, avg_price_by_type.values,
              color=['#ff9999', '#66b3ff', '#99ff99', '#ffcc99', '#c2c2f0'], edgecolor='black')
ax.set_title('Average price by property type (GBP)', fontsize=14)
ax.set_ylabel('Average price (GBP)')
for bar, val in zip(bars, avg_price_by_type.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
            f'{val:,.0f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

---
## Задание 4.2. Визуализация направленных ациклических графов (DAG)

Dask составляет граф выполнения задач (DAG) перед тем, как начать вычисления. Визуализируем графы вычислений, построенные на основе нашего датасета.

### 4.2.1. Визуализация простого DAG

Визуализируем граф вычислений для очищенного датафрейма `df_dropped` — это DAG ETL-процесса (чтение CSV и удаление столбцов). Граф отражает ленивую цепочку операций, которую Dask выполнит при вызове `.compute()`.

In [ ]:
# Визуализация простого DAG - граф ETL-пайплайна
df_dropped.visualize(filename='etl_dag', format='png')

### 4.2.2. Визуализация сложного (многоуровневого) DAG

Визуализируем граф вычислений для операции вычисления средней цены недвижимости. Этот DAG значительно сложнее: он включает приведение типов (`astype`), агрегацию (`mean`) по всем партициям датасета и финальное объединение результатов — это имитация map-reduce процесса на реальных данных.

In [ ]:
# Визуализация сложного DAG - вычисление средней цены по всему датасету
mean_price_task = df['price'].astype(float).mean()
mean_price_task.visualize(filename='mean_price_dag', format='png')

In [ ]:
# Запуск вычислений сложного DAG
with ProgressBar():
    result = mean_price_task.compute()
print(f"Средняя цена недвижимости: {result:,.2f} GBP")

---
## Контрольные вопросы

### 1. В чем главное отличие архитектуры dask.dataframe от классического pandas.dataframe в контексте обработки Big Data?

`pandas.DataFrame` загружает все данные целиком в оперативную память, что ограничивает размер обрабатываемого датасета объемом RAM. `dask.DataFrame` разбивает данные на множество небольших партиций (partitions), каждая из которых является обычным `pandas.DataFrame`. Это позволяет обрабатывать датасеты, значительно превышающие объем оперативной памяти, загружая и обрабатывая данные по частям. Кроме того, Dask поддерживает параллельное выполнение операций над разными партициями на нескольких ядрах или узлах кластера.

### 2. Что такое ленивые вычисления (lazy evaluation) и почему вызов метода `.compute()` следует откладывать на самый конец ETL-пайплайна?

Ленивые вычисления — это стратегия, при которой выражения не вычисляются в момент их определения, а записываются в виде графа задач (DAG). Фактические вычисления запускаются только при вызове `.compute()`. Откладывать `.compute()` до конца пайплайна следует потому, что это позволяет Dask оптимизировать весь граф вычислений целиком: объединять операции, избегать лишних промежуточных материализаций данных в памяти и эффективно распределять нагрузку между воркерами. Ранний вызов `.compute()` приводит к загрузке промежуточных результатов в память, что может вызвать переполнение при работе с большими данными.

### 3. Что такое DAG (Directed Acyclic Graph), как он формируется в Dask и какую роль играет в оптимизации планировщика задач?

DAG (направленный ациклический граф) — это структура данных, в которой узлы представляют задачи (операции над данными), а ребра — зависимости между ними. В Dask DAG формируется автоматически при выполнении ленивых операций: каждый вызов (например, `.drop()`, `.sum()`, `dask.delayed()`) добавляет новые узлы и связи в граф. Планировщик Dask использует DAG для определения оптимального порядка выполнения задач: он выявляет независимые задачи для параллельного выполнения, минимизирует пересылку данных между воркерами и освобождает память от промежуточных результатов, которые больше не нужны.

---
## Вывод

В ходе лабораторной работы был построен полный ETL-конвейер средствами библиотеки Dask, который позволил обработать датасет объемом 4.6 ГБ (~28 млн записей о сделках с недвижимостью в Великобритании) без переполнения оперативной памяти, благодаря механизму ленивых вычислений и разбиению данных на партиции. Анализ данных выявил устойчивый рост средних цен на недвижимость с 1995 по 2023 год, при этом наиболее дорогим типом являются отдельные дома (Detached), а самым распространенным — дома рядовой застройки (Terraced) и двухквартирные дома (Semi-Detached). Визуализация DAG-графов продемонстрировала, как Dask формирует план выполнения задач перед их запуском, что позволяет оптимизировать порядок вычислений и минимизировать потребление ресурсов при работе с большими данными.

In [ ]:
# Завершение работы клиента Dask
client.close()